# P67 — Una base formal para la determinación heurística de caminos de coste mínimo

## 1. Título y paper

**Paper:** *A Formal Basis for the Heuristic Determination of Minimum Cost Paths*  
**Autoría:** Peter E. Hart, Nils J. Nilsson, Bertram Raphael  
**Año y venue:** 1968 · IEEE Transactions on Systems Science and Cybernetics, 4(2), 100–107  
**Nivel:** L3 · **Motor:** `a_estrella`  
**Ficha completa:** [`P67_a_estrella`](../../papers/foundational/P67_a_estrella/README.md)

**Hito:** Convierte la heurística de recurso práctico en garantía demostrable: si nunca sobrestima, el camino encontrado es óptimo.

- [doi:10.1109/TSSC.1968.300136](https://doi.org/10.1109/TSSC.1968.300136)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: La búsqueda guiada por heurística era rápida pero no garantizaba nada. La búsqueda exhaustiva garantizaba optimalidad y no escalaba. No había teoría que uniera las dos.
2. Ejecutar una implementación mínima de la propuesta: Evaluar cada nodo por `f(n) = g(n) + h(n)` —coste acumulado más estimación restante— y demostrar que si `h` es admisible (nunca sobrestima), el algoritmo devuelve el camino de coste mínimo, y que es óptimamente eficiente entre los que usan la misma información.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- Dijkstra (1959), caminos mínimos sin heurística
- P64


## 4. Intuición

Una heurística te dice qué parece prometedor. El problema es que «parece» no es «es»: la búsqueda voraz encuentra rápido y encuentra mal. A* suma las dos mitades —lo que ya llevas gastado y lo que estimas que falta— y demuestra que, si la estimación nunca se pasa, el camino que devuelve es el mejor.


## 5. Concepto mínimo

```text
f(n) = g(n) + h(n)

    g(n) = coste REAL desde el inicio hasta n
    h(n) = coste ESTIMADO desde n hasta la meta

Admisibilidad:  h(n) ≤ coste real restante para todo n
Teorema:        h admisible  ⟹  A* devuelve el camino óptimo
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('a_estrella', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Qué coste tiene el camino óptimo?
2. ¿Qué devuelve la búsqueda voraz, que solo mira h?
3. ¿Y A* con una heurística que sobrestima en un nodo?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('a_estrella', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('a_estrella', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

El óptimo cuesta **8**. La búsqueda voraz expande menos nodos y devuelve un camino de coste **10**: rápida y equivocada. Y A* con la heurística que sobrestima en D —el nodo del camino óptimo— también devuelve **10**. La garantía se pierde exactamente donde se rompe la admisibilidad, y no antes.


## 10. Comentario pedagógico

Lo que aporta el paper no es una heurística mejor: es un **teorema**. Y por eso sobrevive: la búsqueda en árbol de [AlphaGo](../../papers/foundational/P27_alphago/README.md) es esta estructura con una red aportando la estimación, y cualquier planificador de rutas que uses hoy es esto con un mapa detrás.


## 11. Error o anti-patrón deliberado

Anti-patrón: usar una heurística «que funciona bien» sin comprobar que es admisible.


In [ ]:
print('Una heuristica optimista (nunca se pasa) garantiza el camino optimo.')
print('Una que se pasa aunque sea en UN nodo, no garantiza nada.')
print('Y el fallo es silencioso: devuelve un camino, solo que no el mejor.')

## 12. Corrección

La comprobación que hay que hacer, nodo a nodo:


In [ ]:
r = run_paper_lab('a_estrella', seed=7)['result']
print('h admisible en todos los nodos :', r['admisible_en_todos_los_nodos'])
print('la otra falla en              :', r['inadmisible_falla_en'])
for nombre, res in r['resultados'].items():
    print(f"{nombre:<24} coste {res['coste']} · expandidos {res['expandidos']} · optimo {res['es_optimo']}")

## 13. Desafío guiado

Compara los nodos expandidos por costo uniforme y por A* admisible, y explica por qué A* expande menos sin perder la garantía.


In [ ]:
r = run_paper_lab('a_estrella', seed=3)['result']
show(r)

## 14. Desafío autónomo

Implementa A* sobre una rejilla con obstáculos usando la distancia Manhattan, comprueba que es admisible y después multiplícala por 1,5. Mide cuánto ganas en nodos y cuánto pierdes en calidad del camino.


## 15. Evidencia de aprendizaje

Guarda la tabla de las cuatro búsquedas con coste, nodos y optimalidad, y tu comprobación de admisibilidad nodo a nodo.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P67_a_estrella/README.md) · evaluación formal: [`assessments/papers/P67_a_estrella.md`](../../assessments/papers/P67_a_estrella.md)


## 16. Cierre

La búsqueda ya tiene garantía. Pero para planificar hace falta algo más: una forma de describir acciones que no obligue a decir todo lo que NO cambia.


## 17. Conexión con el siguiente hito

- P27
- P68

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
